# 04. Long-run balance decomposition

Read the canonical decomposition over 1977-2025, identify the years with a positive aggregate balance, and quantify the Social Security offset metrics as accounting ratios.

**Reads**

- `data/processed/annual_balance_metrics_1977_2025.csv`
- `outputs/metrics/analysis_summary.json`

**Writes**

- Nothing. All metrics shown here are persisted by the pipeline.

**Method reference:** `METHODOLOGY.md` section 5

In [ ]:
"""Notebook environment: locate the repository and expose its data layers."""

import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

# Resolve the repository root from wherever the kernel was started, so the
# notebook works both from the repository root and from the notebooks directory.
ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

%matplotlib inline

from portugal_fiscal_balance.analysis import figures

RAW = ROOT / 'data' / 'raw'
INTERIM = ROOT / 'data' / 'interim'
PROCESSED = ROOT / 'data' / 'processed'
TABLES = ROOT / 'outputs' / 'tables'
METRICS = ROOT / 'outputs' / 'metrics'

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

print('repository:', ROOT.name)
print('pipeline outputs present:', (PROCESSED / 'fiscal_balances_1977_2025.csv').exists())

## 1. The panel

`annual_balance_metrics_1977_2025.csv` is the canonical balance panel plus derived
offset and contribution metrics. Every column is calculated; none is transcribed.

In [ ]:
annual = pd.read_csv(PROCESSED / 'annual_balance_metrics_1977_2025.csv')
levels = [
    'year',
    'general_government_balance_m_eur',
    'central_government_balance_m_eur',
    'regional_local_balance_m_eur',
    'social_security_balance_m_eur',
    'non_ssf_balance_m_eur',
    'ssf_offset_ratio',
]
print('years:', int(annual['year'].min()), 'to', int(annual['year'].max()), f'({len(annual)} observations)')
display(annual[levels].tail(12).round(3))

In [ ]:
figure = figures.balances_by_subsector(annual)

## 2. Contributions to the aggregate

The stacked columns below are the three subsector balances; their signed sum is
the General Government line. The chart is the identity drawn, so a tall column
under a shallow line means the subsectors offset one another that year.

In [ ]:
figure = figures.subsector_contributions(annual)

## 3. Positive aggregate-balance years

$$B^{nonSSF}_t = B^{C}_t + B^{RL}_t$$

and, when $B^{nonSSF}_t < 0$ and $B^{SSF}_t > 0$, the offset ratio is

$$O_t = \frac{B^{SSF}_t}{\left|B^{nonSSF}_t\right|}.$$

$O_t = 1$ means the positive Social Security balance is exactly the size of the
negative non-SSF balance. The ratio is an accounting comparison of two recorded
numbers. It is not a counterfactual, and it does not describe what either balance
would be under different institutional arrangements.

In [ ]:
positive = annual.loc[
    annual['aggregate_balance_positive'],
    [
        'year',
        'general_government_balance_m_eur',
        'non_ssf_balance_m_eur',
        'social_security_balance_m_eur',
        'ssf_offset_ratio',
        'ssf_share_of_positive_aggregate_balance',
    ],
]
display(positive.round(3))
print('years with a positive aggregate balance:', [int(year) for year in positive['year']])

In [ ]:
figure = figures.offset_ratio(annual)

## 4. Regime-aware averages

Five-year rolling means are shown because single-year balances are volatile. The
window spans the 1995 splice for the years around it, so those values mix two
statistical vintages and should be read accordingly.

In [ ]:
rolling = [
    'year',
    'general_government_balance_5y_mean_pct_gdp',
    'central_government_balance_5y_mean_pct_gdp',
    'regional_local_balance_5y_mean_pct_gdp',
    'social_security_balance_5y_mean_pct_gdp',
]
display(annual[rolling].tail(10).round(3))

In [ ]:
import json

summary = json.loads((METRICS / 'analysis_summary.json').read_text(encoding='utf-8'))
latest = summary['balance_summary']['latest_year']
print('latest year in the panel:', latest['year'])
display(
    pd.Series({key: value for key, value in latest.items() if key != 'year'}, name=str(latest['year']))
    .to_frame()
    .round(3)
)

## Interpretation limits

1. The offset ratio is **defined only** when the non-SSF balance is negative and
   the Social Security balance is positive. It is `NaN` otherwise, by design.
2. A subsector's arithmetic contribution to a positive aggregate balance is
   **not evidence of causation** and not a statement about policy intent.
3. The 1977-1994 and 1995-2025 segments come from **different statistical
   vintages**. Long-run comparisons carry that caveat throughout.

---

[Previous: 03. Harmonisation and validation](03_harmonize_and_validate.ipynb) | [Next: 05. Revenue and expenditure decomposition](05_revenue_expenditure.ipynb)

Every table shown above is also persisted as CSV, so results can be checked without reading notebook state. To rebuild everything from the bundled raw sources:

```bash
poetry install
make all
```